### Process Orders data
1. Ingest the data into the data lakehouse- bronze_orders
2. Perform data quality checks and transform the data as required - silver_orders_clean
3. Explode the items array from the order object - silver_orders

## 1.Ingest the data into the data lakehouse- bronze_orders

![image_1781008922447.png](./image_1781008922447.png "image_1781008922447.png")

In [0]:
import dlt
from pyspark.sql.functions import *

In [0]:
@dlt.table(
    name="bronze_orders",
    table_properties={"quality":"bronze"},
    comment="Raw Orders data ingested from the source system"
)
def create_bronze_orders():
    return(
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes",True)
            .load("/Volumes/catalog_circuitbox/landing/operational_data/orders/")
    )

## 2. Perform data quality checks and transform the data as required - silver_orders_clean


![image_1781008899720.png](./image_1781008899720.png "image_1781008899720.png")

In [0]:
@dlt.table(
    name="silver_orders_clean",
    table_properties={'quality':'silver'},
    comment="Cleaned Order data"
)
@dlt.expect_or_fail("valid_customer_id","customer_id IS NOT NULL")
@dlt.expect_or_fail("valid_order_id","order_id IS NOT NULL")
@dlt.expect("valiad_order_status", "order_status IN ('Pending','Shipped','Cancelled','Completed') ")
@dlt.expect("valid_payment_method", "payment_method IN ('Creadit Card','Bank Transfer','PayPal') ")
def silver_orders_clean():
    return (
        spark.readStream.table("LIVE.bronze_orders")
            .select(
                col("order_id"),
                col("customer_id"),
                col("order_timestamp").cast("timestamp").alias("order_timestamp"),
                col("payment_method"),
                col("order_status"),
                col("items")
        )
    )         